# Data Splitting in Machine Learning
This notebook demonstrates core data splitting strategies used before training ML models.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold

# Load a sample dataset
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name='species')

print(f"Dataset shape: {X.shape}")
print(f"Class distribution:\n{y.value_counts().to_string()}")

## 1. Train / Test Split (Hold-out)
The simplest approach — reserve a portion of data for final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% held out for testing
    random_state=42,
    stratify=y          # preserve class proportions
)

print(f"Training set : {len(X_train)} samples ({len(X_train)/len(X):.0%})")
print(f"Test set     : {len(X_test)}  samples ({len(X_test)/len(X):.0%})")
print(f"\nClass balance in training set:\n{y_train.value_counts().to_string()}")
print(f"\nClass balance in test set:\n{y_test.value_counts().to_string()}")

## 2. Train / Validation / Test Split
Three-way split — training, tuning (validation), and final evaluation (test).

In [ ]:
# Step 1: carve off test set (20%)
X_temp, X_test2, y_temp, y_test2 = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Step 2: split the remainder into train (75%) and validation (25%) → 60/20 overall
X_train2, X_val, y_train2, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"Train      : {len(X_train2):>4} samples ({len(X_train2)/len(X):.0%})")
print(f"Validation : {len(X_val):>4} samples ({len(X_val)/len(X):.0%})")
print(f"Test       : {len(X_test2):>4} samples ({len(X_test2)/len(X):.0%})")

## 3. K-Fold Cross-Validation
Rotates which fold acts as the validation set — every sample is used for both training and validation.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("K-Fold splits (indices shown for first 5 test samples per fold)\n")
for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f"  Fold {fold}: train={len(train_idx)} samples | val={len(val_idx)} samples | "
          f"val indices (first 5): {val_idx[:5].tolist()}")

## 4. Stratified K-Fold
Like K-Fold but preserves the class distribution in every fold — essential for imbalanced datasets.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Stratified K-Fold — class counts per fold\n")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    counts = y.iloc[val_idx].value_counts().sort_index()
    print(f"  Fold {fold}: val size={len(val_idx)} | classes → {dict(counts)}")

## 5. Visualisation — Split Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Data Splitting Strategies", fontsize=14, fontweight='bold')

# ── Left: proportions bar chart ──────────────────────────────────────────────
splits = {
    "Train/Test\n(80/20)": [80, 0, 20],
    "Train/Val/Test\n(60/20/20)": [60, 20, 20],
}
colors = ['#4C9BE8', '#F4A460', '#E85C5C']
labels = ['Train', 'Validation', 'Test']
x = np.arange(len(splits))
bottom = np.zeros(len(splits))

for i, (label, color) in enumerate(zip(labels, colors)):
    vals = [splits[k][i] for k in splits]
    bars = axes[0].bar(x, vals, bottom=bottom, color=color, label=label, width=0.5)
    for bar, v in zip(bars, vals):
        if v > 0:
            axes[0].text(
                bar.get_x() + bar.get_width()/2,
                bar.get_y() + bar.get_height()/2,
                f"{v}%", ha='center', va='center', fontsize=11, color='white', fontweight='bold'
            )
    bottom += np.array(vals)

axes[0].set_xticks(x)
axes[0].set_xticklabels(splits.keys(), fontsize=10)
axes[0].set_ylabel("Percentage of data")
axes[0].set_title("Hold-out Splits")
axes[0].legend(loc='upper right')
axes[0].set_ylim(0, 105)

# ── Right: K-Fold tile diagram ────────────────────────────────────────────────
n_folds = 5
ax = axes[1]
for fold in range(n_folds):
    for segment in range(n_folds):
        color = '#E85C5C' if segment == fold else '#4C9BE8'
        rect = mpatches.FancyBboxPatch(
            (segment * 1.05, n_folds - fold - 1),
            0.95, 0.85,
            boxstyle="round,pad=0.03",
            facecolor=color, edgecolor='white', linewidth=1.5
        )
        ax.add_patch(rect)
        ax.text(segment * 1.05 + 0.475, n_folds - fold - 0.575,
                f"F{segment+1}", ha='center', va='center',
                fontsize=9, color='white', fontweight='bold')
    ax.text(-0.6, n_folds - fold - 0.575, f"Fold {fold+1}",
            ha='center', va='center', fontsize=9)

ax.set_xlim(-1, n_folds * 1.05)
ax.set_ylim(-0.3, n_folds)
ax.axis('off')
ax.set_title("5-Fold Cross-Validation")
train_patch = mpatches.Patch(color='#4C9BE8', label='Train')
val_patch   = mpatches.Patch(color='#E85C5C', label='Validation')
ax.legend(handles=[train_patch, val_patch], loc='lower right')

plt.tight_layout()
plt.savefig('split_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved.")

## Summary

| Strategy | When to use |
|---|---|
| **Train/Test** | Large datasets where a held-out test set is sufficient |
| **Train/Val/Test** | Hyperparameter tuning — val set guides tuning, test set is untouched |
| **K-Fold CV** | Small/medium datasets — maximises data usage, reduces variance in metric estimates |
| **Stratified K-Fold** | Imbalanced classes — ensures each fold reflects the overall class distribution |

> **Golden rule:** the test set must never be seen during training *or* tuning — look at it only once, at the very end.